# Réseau neuronal convolutif (CNN)

In [48]:
import numpy as np
import copy
import matplotlib.pyplot as plt
import h5py
import scipy
from PIL import Image
from scipy import ndimage

%matplotlib inline
plt.rcParams['figure.figsize'] = (5.0, 4.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

%load_ext autoreload
%autoreload 2

np.random.seed(1)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [49]:
## Convolutional layer
import numpy as np
import scipy.signal

def conv_forward(A_prev, W, b, stride=1, padding=0):
    (m, n_H_prev, n_W_prev, n_C_prev) = A_prev.shape
    (f, f, n_C_prev, n_C) = W.shape
    n_H = int((n_H_prev + 2 * padding - f) / stride) + 1
    n_W = int((n_W_prev + 2 * padding - f) / stride) + 1

    A_prev_pad = np.pad(A_prev, ((0, 0), (padding, padding), (padding, padding), (0, 0)), mode='constant', constant_values=(0, 0))
    Z = np.zeros((m, n_H, n_W, n_C))

    for i in range(m):
        a_prev_pad = A_prev_pad[i]
        for h in range(n_H):
            for w in range(n_W):
                for c in range(n_C):
                    vert_start = h * stride
                    vert_end = vert_start + f
                    horiz_start = w * stride
                    horiz_end = horiz_start + f
                    a_slice = a_prev_pad[vert_start:vert_end, horiz_start:horiz_end, :]
                    Z[i, h, w, c] = np.sum(a_slice * W[:, :, :, c]) + b[:, :, :, c]
    return Z

def conv_backward(dZ, A_prev, W, stride=1, padding=0):
    (m, n_H_prev, n_W_prev, n_C_prev) = A_prev.shape
    (f, f, n_C_prev, n_C) = W.shape
    (m, n_H, n_W, n_C) = dZ.shape

    A_prev_pad = np.pad(A_prev, ((0, 0), (padding, padding), (padding, padding), (0, 0)), mode='constant', constant_values=(0, 0))
    dA_prev_pad = np.zeros_like(A_prev_pad)
    dW = np.zeros_like(W)
    db = np.zeros_like(b)

    for i in range(m):
        a_prev_pad = A_prev_pad[i]
        da_prev_pad = dA_prev_pad[i]
        for h in range(n_H):
            for w in range(n_W):
                for c in range(n_C):
                    vert_start = h * stride
                    vert_end = vert_start + f
                    horiz_start = w * stride
                    horiz_end = horiz_start + f
                    a_slice = a_prev_pad[vert_start:vert_end, horiz_start:horiz_end, :]
                    da_prev_pad[vert_start:vert_end, horiz_start:horiz_end, :] += W[:, :, :, c] * dZ[i, h, w, c]
                    dW[:, :, :, c] += a_slice * dZ[i, h, w, c]
                    db[:, :, :, c] += dZ[i, h, w, c]
        dA_prev_pad[i] = da_prev_pad
    dA_prev = dA_prev_pad[:, padding:-padding, padding:-padding, :]
    return dA_prev, dW, db

In [50]:
## Pooling layer
def pool_forward(A_prev, f=2, stride=2, mode="max"):
    (m, n_H_prev, n_W_prev, n_C_prev) = A_prev.shape
    n_H = int(1 + (n_H_prev - f) / stride)
    n_W = int(1 + (n_W_prev - f) / stride)
    n_C = n_C_prev

    A = np.zeros((m, n_H, n_W, n_C))

    for i in range(m):
        for h in range(n_H):
            for w in range(n_W):
                for c in range(n_C):
                    vert_start = h * stride
                    vert_end = vert_start + f
                    horiz_start = w * stride
                    horiz_end = horiz_start + f
                    a_slice = A_prev[i, vert_start:vert_end, horiz_start:horiz_end, c]

                    if mode == "max":
                        A[i, h, w, c] = np.max(a_slice)
                    elif mode == "average":
                        A[i, h, w, c] = np.mean(a_slice)
    return A

def pool_backward(dA, A_prev, f=2, stride=2, mode="max"):
    (m, n_H_prev, n_W_prev, n_C_prev) = A_prev.shape
    (m, n_H, n_W, n_C) = dA.shape

    dA_prev = np.zeros_like(A_prev)

    for i in range(m):
        a_prev = A_prev[i]
        for h in range(n_H):
            for w in range(n_W):
                for c in range(n_C):
                    vert_start = h * stride
                    vert_end = vert_start + f
                    horiz_start = w * stride
                    horiz_end = horiz_start + f

                    if mode == "max":
                        a_slice = a_prev[vert_start:vert_end, horiz_start:horiz_end, c]
                        mask = a_slice == np.max(a_slice)
                        dA_prev[i, vert_start:vert_end, horiz_start:horiz_end, c] += mask * dA[i, h, w, c]
                    elif mode == "average":
                        da = dA[i, h, w, c]
                        shape = (f, f)
                        dA_prev[i, vert_start:vert_end, horiz_start:horiz_end, c] += distribute_value(da, shape)
    return dA_prev

def distribute_value(dz, shape):
    (n_H, n_W) = shape
    average = dz / (n_H * n_W)
    return np.ones(shape) * average

In [51]:
## Fully connected layer
def fc_forward(A, W, b):
    Z = np.dot(A, W) + b
    return Z

def fc_backward(dZ, A, W, b):
    m = A.shape[0]
    dW = np.dot(A.T, dZ) / m
    db = np.sum(dZ, axis=0, keepdims=True) / m
    dA = np.dot(dZ, W.T)
    return dA, dW, db

In [52]:
## Model CNN
class SimpleCNN:
    def __init__(self):
        self.parameters = self.initialize_parameters()

    def initialize_parameters(self):
        np.random.seed(1)
        parameters = {}
        parameters['W1'] = np.random.randn(3, 3, 1, 8) * 0.1
        parameters['b1'] = np.zeros((1, 1, 1, 8))
        parameters['W2'] = np.random.randn(3, 3, 8, 16) * 0.1
        parameters['b2'] = np.zeros((1, 1, 1, 16))
        parameters['W3'] = np.random.randn(400, 128) * 0.1
        parameters['b3'] = np.zeros((1, 128))
        parameters['W4'] = np.random.randn(128, 1) * 0.1
        parameters['b4'] = np.zeros((1, 1))
        return parameters

    def forward_propagation(self, X):
        parameters = self.parameters
        self.cache = {}

        # Conv Layer 1
        self.cache['Z1'] = conv_forward(X, parameters['W1'], parameters['b1'])
        self.cache['A1'] = np.maximum(0, self.cache['Z1'])  # ReLU Activation
        self.cache['P1'] = pool_forward(self.cache['A1'])

        # Conv Layer 2
        self.cache['Z2'] = conv_forward(self.cache['P1'], parameters['W2'], parameters['b2'])
        self.cache['A2'] = np.maximum(0, self.cache['Z2'])  # ReLU Activation
        self.cache['P2'] = pool_forward(self.cache['A2'])

        # Flatten
        self.cache['F'] = self.cache['P2'].reshape(self.cache['P2'].shape[0], -1)

        # Fully Connected Layer 1
        self.cache['Z3'] = fc_forward(self.cache['F'], parameters['W3'], parameters['b3'])
        self.cache['A3'] = np.maximum(0, self.cache['Z3'])  # ReLU Activation

        # Fully Connected Layer 2 (Output Layer)
        self.cache['Z4'] = fc_forward(self.cache['A3'], parameters['W4'], parameters['b4'])
        self.cache['A4'] = 1 / (1 + np.exp(-self.cache['Z4']))  # Sigmoid Activation

        return self.cache['A4']

    def backward_propagation(self, X, Y):
        parameters = self.parameters
        cache = self.cache

        m = X.shape[0]

        # Output layer
        dZ4 = cache['A4'] - Y
        dA3, dW4, db4 = fc_backward(dZ4, cache['A3'], parameters['W4'], parameters['b4'])

        # Fully connected layer 1
        dA3[cache['Z3'] <= 0] = 0  # ReLU backpropagation
        dA2_flat, dW3, db3 = fc_backward(dA3, cache['F'], parameters['W3'], parameters['b3'])

        # Reshape gradient to match P2 dimensions
        dA2 = dA2_flat.reshape(cache['P2'].shape)

        # Pooling layer 2
        dP2 = pool_backward(dA2, cache['A2'])
        dP2[cache['Z2'] <= 0] = 0  # ReLU backpropagation

        # Convolutional layer 2
        dA1, dW2, db2 = conv_backward(dP2, cache['P1'], parameters['W2'])

        # Pooling layer 1
        dP1 = pool_backward(dA1, cache['A1'])
        dP1[cache['Z1'] <= 0] = 0  # ReLU backpropagation

        # Convolutional layer 1
        dA_prev, dW1, db1 = conv_backward(dP1, X, parameters['W1'])

        # Update gradients
        grads = {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2, 'dW3': dW3, 'db3': db3, 'dW4': dW4, 'db4': db4}
        return grads

    def update_parameters(self, grads, learning_rate):
        for key in self.parameters.keys():
            self.parameters[key] -= learning_rate * grads['d' + key]

    def compute_cost(self, A4, Y):
        m = Y.shape[0]
        cost = -np.sum(Y * np.log(A4) + (1 - Y) * np.log(1 - A4)) / m
        cost = np.squeeze(cost)
        return cost

    def train(self, X, Y, learning_rate=0.01, num_iterations=100):
        for i in range(num_iterations):
            A4 = self.forward_propagation(X)
            cost = self.compute_cost(A4, Y)
            grads = self.backward_propagation(X, Y)
            self.update_parameters(grads, learning_rate)
            if i % 10 == 0:
                print(f"Cost after iteration {i}: {cost}")

# Example of using this model

In [53]:
def load_dataset():
    train_dataset = h5py.File('datasets/train_catvnoncat.h5', "r")
    train_set_x_orig = np.array(train_dataset["train_set_x"][:]) # your train set features
    train_set_y_orig = np.array(train_dataset["train_set_y"][:]) # your train set labels

    test_dataset = h5py.File('datasets/test_catvnoncat.h5', "r")
    test_set_x_orig = np.array(test_dataset["test_set_x"][:]) # your test set features
    test_set_y_orig = np.array(test_dataset["test_set_y"][:]) # your test set labels

    classes = np.array(test_dataset["list_classes"][:]) # the list of classes
    
    train_set_y_orig = train_set_y_orig.reshape((1, train_set_y_orig.shape[0]))
    test_set_y_orig = test_set_y_orig.reshape((1, test_set_y_orig.shape[0]))
    
    return train_set_x_orig, train_set_y_orig, test_set_x_orig, test_set_y_orig, classes

In [54]:
# Chargement des données (chat/non-chat)
train_set_x_orig, train_set_y, test_set_x_orig, test_set_y, classes = load_dataset()

n_train = train_set_x_orig.shape[0]
n_test = test_set_x_orig.shape[0]
num_px = train_set_x_orig.shape[1]  

train_set_x_flatten = train_set_x_orig.reshape(train_set_x_orig.shape[0], -1).T
test_set_x_flatten = test_set_x_orig.reshape(test_set_x_orig.shape[0], -1).T 

train_set_x = train_set_x_flatten / 255.
test_set_x = test_set_x_flatten / 255.

In [57]:
#X = train_set_x_orig
#Y = train_set_y
X = np.random.randn(10, 64, 64, 1)  # 10个64x64的灰度图像
Y = np.random.randint(0, 2, (10, 1))  # 10个二进制标签

cnn = SimpleCNN()
cnn.train(X, Y, learning_rate=0.01, num_iterations=100)

ValueError: shapes (10,3136) and (400,128) not aligned: 3136 (dim 1) != 400 (dim 0)